In [ ]:
# ============================================================
# CELL 1 — ALL DATA / FROZEN STRATEGY / EVOLUTIONARY SEARCH
# ============================================================

# ============================================================
# CELL 1 — V8 FROZEN STRATEGY CONFIG
# ============================================================

from QuantConnect.Research import QuantBook
from QuantConnect import Resolution, Market

import pandas as pd
import numpy as np
from datetime import datetime

qb = QuantBook()

def _market_attr(*names):
    for name in names:
        if hasattr(Market, name):
            return getattr(Market, name)
    raise AttributeError(names)

OANDA = _market_attr("Oanda", "OANDA")

DATA_START = datetime(2022, 1, 1)
DATA_END   = datetime(2026, 8, 15)

ASSETS = {
    "EURUSD": {
        "type": "forex",
        "market": OANDA,
        "maker_bps": 0.0,
        "taker_bps": 0.0,
        "fallback_spread_bps": 0.0,
    },
    "NAS100USD": {
        "type": "cfd",
        "market": OANDA,
        "maker_bps": 0.0,
        "taker_bps": 0.0,
        "fallback_spread_bps": 0.0,
    },
}

TIMEFRAME_MIN = 60
STRUCTURE_SIZE = 5
ATR_PERIOD = 14

# ---------------- FROZEN V5 SIGNAL ----------------
FROZEN_MIN_DISPLACEMENT_ATR = 2.0
FROZEN_MIN_BROKEN_STRUCTURE_ATR = 1.0
FROZEN_MIN_CLOSE_THROUGH_ATR = 0.25
RETRACE = 0.618
PENDING_EXPIRY_BARS = 20
MAX_ENTRY_DISTANCE_ATR = 2.0
MAX_HOLD_BARS = 120   # allow runners more breathing room

INTRABAR_PRIORITY = "stop_first"
CANCEL_ON_NEW_CHOCH = True
CANCEL_ON_OPPOSITE_BOS = True
INVALIDATE_ON_ORIGIN_BREAK = True

# Exit policies only.
NORMAL_TP_POLICIES = [
    ("structural", None),
    ("fixed_r", 1.5),
    ("fixed_r", 2.0),
    ("fixed_r", 3.0),
]

RUNNER_SPLITS = [
    (0.90, 0.10),
    (0.80, 0.20),
]

RUNNER_TARGETS_R = [5.0, 10.0, 20.0]

SLIPPAGE_STRESS_BPS = [0.0, 0.5, 1.0]

print("V7 configured.")
print("Signal frozen.")
print("Normal exits:", NORMAL_TP_POLICIES)
print("Runner splits:", RUNNER_SPLITS)
print("Runner targets:", RUNNER_TARGETS_R)

# V8: exit is now frozen at 1.5R. Portfolio research must not alter signal/entry/SL/TP.
FROZEN_TP_R = 1.5
PORTFOLIO_RISK_LEVELS = [0.0025, 0.0050]
SELECTION_END_YEAR = 2025
HOLDOUT_YEAR = 2026


# ============================================================
# CELL 2 — DATA SUBSCRIPTIONS + PANDAS RESAMPLING
# ============================================================

SECURITIES = {}

def add_asset(pair, spec):
    if spec["type"] == "forex":
        return qb.AddForex(pair, Resolution.Minute, spec["market"]).Symbol
    if spec["type"] == "cfd":
        return qb.AddCfd(pair, Resolution.Minute, spec["market"]).Symbol
    if spec["type"] == "crypto":
        return qb.AddCrypto(pair, Resolution.Minute, spec["market"]).Symbol
    raise ValueError(spec["type"])

for pair, spec in ASSETS.items():
    try:
        SECURITIES[pair] = add_asset(pair, spec)
        print(pair, "subscribed")
    except Exception as exc:
        print(pair, "subscription failed:", exc)

def _flatten_history(history, pair):
    if history is None or history.empty:
        return pd.DataFrame()

    df = history.copy()

    if isinstance(df.index, pd.MultiIndex):
        try:
            df = df.loc[pair].copy()
        except Exception:
            df = df.reset_index()
            time_col = next((c for c in ["time","endtime","index"] if c in df.columns), None)
            if time_col is None:
                raise
            df = df.set_index(time_col)

    df.columns = [str(c).lower().replace("_","") for c in df.columns]

    idx = pd.to_datetime(df.index)
    if idx.tz is None:
        idx = idx.tz_localize("UTC")
    else:
        idx = idx.tz_convert("UTC")

    df.index = idx
    return df.sort_index()

def normalize_ohlc(history, pair):
    raw = _flatten_history(history, pair)
    if raw.empty:
        return raw

    out = pd.DataFrame(index=raw.index)

    bid = ["bidopen","bidhigh","bidlow","bidclose"]
    ask = ["askopen","askhigh","asklow","askclose"]

    has_quotes = all(c in raw.columns for c in bid + ask)

    if has_quotes:
        for c in bid + ask:
            out[c] = pd.to_numeric(raw[c], errors="coerce")

        out["open"]  = (out["bidopen"]  + out["askopen"])  / 2
        out["high"]  = (out["bidhigh"]  + out["askhigh"])  / 2
        out["low"]   = (out["bidlow"]   + out["asklow"])   / 2
        out["close"] = (out["bidclose"] + out["askclose"]) / 2
    else:
        for c in ["open","high","low","close"]:
            out[c] = pd.to_numeric(raw[c], errors="coerce")

    return out.dropna(subset=["open","high","low","close"])

def resample_ohlc(df, minutes):
    agg = {
        "open":"first",
        "high":"max",
        "low":"min",
        "close":"last",
    }

    quote_cols = {
        "bidopen":"first","bidhigh":"max","bidlow":"min","bidclose":"last",
        "askopen":"first","askhigh":"max","asklow":"min","askclose":"last",
    }

    for c, how in quote_cols.items():
        if c in df.columns:
            agg[c] = how

    return (
        df.resample(f"{minutes}min")
        .agg(agg)
        .dropna(subset=["open","high","low","close"])
    )

DATA_1M = {}
DATA = {}

for pair, symbol in SECURITIES.items():
    try:
        hist = qb.History(symbol, DATA_START, DATA_END, Resolution.Minute)
        df1 = normalize_ohlc(hist, pair)

        if df1.empty:
            print(pair, "no data")
            continue

        DATA_1M[pair] = df1
        DATA[pair] = resample_ohlc(df1, TIMEFRAME_MIN)

        print(
            pair,
            f"1m={len(df1):,}",
            f"{TIMEFRAME_MIN}m={len(DATA[pair]):,}"
        )

    except Exception as exc:
        print(pair, "DATA LOAD FAILED:", exc)

print("Loaded:", list(DATA))


# ============================================================
# CELL 3 — ATR + CAUSAL CHoCH/BOS DETECTION
# ============================================================

def add_atr(df, period=14):
    x = df.copy()

    prev = x["close"].shift(1)

    tr = pd.concat(
        [
            x["high"] - x["low"],
            (x["high"] - prev).abs(),
            (x["low"] - prev).abs(),
        ],
        axis=1,
    ).max(axis=1)

    x["atr"] = tr.rolling(
        period,
        min_periods=period
    ).mean()

    return x

def detect_structure_events(df, size):

    x = add_atr(df, ATR_PERIOD)

    if len(x) <= size + 2:
        return pd.DataFrame()

    highs = x["high"].to_numpy(float)
    lows = x["low"].to_numpy(float)
    closes = x["close"].to_numpy(float)
    times = x.index
    atrs = x["atr"].to_numpy(float)

    h = x["high"]
    l = x["low"]

    # Pivot at i-size only becomes knowable at current bar i.
    new_leg_high = (
        h.shift(size) > h.rolling(size).max()
    ).fillna(False).to_numpy(bool)

    new_leg_low = (
        l.shift(size) < l.rolling(size).min()
    ).fillna(False).to_numpy(bool)

    BEARISH_LEG = 0
    BULLISH_LEG = 1
    BEARISH = -1
    BULLISH = 1

    leg = BEARISH_LEG
    trend = 0

    pivot_high = None
    pivot_low = None

    pivot_high_crossed = True
    pivot_low_crossed = True

    events = []

    for i in range(size, len(x)):

        previous_leg = leg

        if new_leg_high[i]:
            leg = BEARISH_LEG
        elif new_leg_low[i]:
            leg = BULLISH_LEG

        if leg != previous_leg:
            src = i - size

            if leg == BULLISH_LEG:
                pivot_low = {
                    "price": float(lows[src]),
                    "pivot_time": times[src],
                    "available_time": times[i],
                }
                pivot_low_crossed = False

            else:
                pivot_high = {
                    "price": float(highs[src]),
                    "pivot_time": times[src],
                    "available_time": times[i],
                }
                pivot_high_crossed = False

        previous_close = closes[i - 1]
        close = closes[i]

        # Bullish break
        if pivot_high is not None and not pivot_high_crossed:

            level = pivot_high["price"]

            if previous_close <= level and close > level:

                tag = "CHoCH" if trend == BEARISH else "BOS"
                trend = BULLISH
                pivot_high_crossed = True

                if pivot_low is not None and pivot_low["available_time"] <= times[i]:

                    origin = float(pivot_low["price"])
                    extreme = float(highs[i])
                    atr = float(atrs[i]) if not np.isnan(atrs[i]) else np.nan

                    events.append({
                        "event_time": times[i],
                        "bar_pos": i,
                        "type": tag,
                        "direction": "bullish",
                        "broken_level": level,
                        "origin_price": origin,
                        "event_extreme": extreme,
                        "atr": atr,
                        "break_strength_atr": (
                            (close - level) / atr
                            if atr and atr > 0
                            else np.nan
                        ),
                    })

        # Bearish break
        if pivot_low is not None and not pivot_low_crossed:

            level = pivot_low["price"]

            if previous_close >= level and close < level:

                tag = "CHoCH" if trend == BULLISH else "BOS"
                trend = BEARISH
                pivot_low_crossed = True

                if pivot_high is not None and pivot_high["available_time"] <= times[i]:

                    origin = float(pivot_high["price"])
                    extreme = float(lows[i])
                    atr = float(atrs[i]) if not np.isnan(atrs[i]) else np.nan

                    events.append({
                        "event_time": times[i],
                        "bar_pos": i,
                        "type": tag,
                        "direction": "bearish",
                        "broken_level": level,
                        "origin_price": origin,
                        "event_extreme": extreme,
                        "atr": atr,
                        "break_strength_atr": (
                            (level - close) / atr
                            if atr and atr > 0
                            else np.nan
                        ),
                    })

    ev = pd.DataFrame(events)

    if not ev.empty:
        ev["initial_displacement_atr"] = np.where(
            ev["direction"] == "bullish",
            (ev["event_extreme"] - ev["origin_price"]) / ev["atr"],
            (ev["origin_price"] - ev["event_extreme"]) / ev["atr"],
        )

        ev["broken_structure_size"] = (
            ev["broken_level"] - ev["origin_price"]
        ).abs()

        ev["broken_structure_atr"] = (
            ev["broken_structure_size"]
            / ev["atr"].replace(0, np.nan)
        )

    return ev

EVENTS = {}

for pair, df in DATA.items():
    ev = detect_structure_events(df, STRUCTURE_SIZE)
    EVENTS[pair] = ev

    counts = ev["type"].value_counts().to_dict() if not ev.empty else {}

    print(
        pair,
        "events=", len(ev),
        "CHoCH=", counts.get("CHoCH",0),
        "BOS=", counts.get("BOS",0)
    )


# ============================================================
# CELL 4 — SUPERSESSION MAPS
# ============================================================

def build_supersession_maps(events):
    if events is None or events.empty:
        return {}

    e = events.sort_values("event_time").reset_index(drop=True)
    out = {}

    for i, row in e.iterrows():
        next_choch = None
        next_opposite_bos = None

        for j in range(i + 1, len(e)):
            r = e.iloc[j]

            if next_choch is None and r["type"] == "CHoCH":
                next_choch = r["event_time"]

            if (
                next_opposite_bos is None
                and r["type"] == "BOS"
                and r["direction"] != row["direction"]
            ):
                next_opposite_bos = r["event_time"]

            if next_choch is not None and next_opposite_bos is not None:
                break

        out[int(row["bar_pos"])] = {
            "next_choch": next_choch,
            "next_opposite_bos": next_opposite_bos,
        }

    return out

SUPERSESSION = {
    pair: build_supersession_maps(ev)
    for pair, ev in EVENTS.items()
}

print("Supersession maps built.")


# ============================================================
# CELL 5 — EXECUTION HELPERS
# ============================================================

def retracement_price(origin, extreme, direction, retrace):
    if direction == "bullish":
        return extreme - retrace * (extreme - origin)

    return extreme + retrace * (origin - extreme)

def pending_limit_filled(bar, direction, limit_price, fallback_spread_bps):
    half = fallback_spread_bps / 20000.0

    if direction == "bullish":

        if "asklow" in bar.index:
            return float(bar["asklow"]) <= limit_price

        return float(bar["low"]) * (1 + half) <= limit_price

    if "bidhigh" in bar.index:
        return float(bar["bidhigh"]) >= limit_price

    return float(bar["high"]) * (1 - half) >= limit_price

def executable_side(bar, direction):

    if direction == "bullish" and "bidlow" in bar.index:
        return {
            "high": float(bar["bidhigh"]),
            "low": float(bar["bidlow"]),
            "close": float(bar["bidclose"]),
        }

    if direction == "bearish" and "asklow" in bar.index:
        return {
            "high": float(bar["askhigh"]),
            "low": float(bar["asklow"]),
            "close": float(bar["askclose"]),
        }

    return {
        "high": float(bar["high"]),
        "low": float(bar["low"]),
        "close": float(bar["close"]),
    }

def fee_in_r(entry_price, exit_price, risk, entry_bps, exit_bps):
    if risk <= 0:
        return 0.0

    fee_price = (
        entry_price * entry_bps / 10000.0
        + exit_price * exit_bps / 10000.0
    )

    return fee_price / risk


# ============================================================
# CELL 6 — DYNAMIC CLOSE-ONLY FIB SIMULATOR
# ============================================================

def simulate_dynamic_trade(
    df,
    event,
    asset_spec,
    supersession_info,
    min_displacement_atr,
    retrace=RETRACE,
    pending_expiry_bars=PENDING_EXPIRY_BARS,
    max_entry_distance_atr=MAX_ENTRY_DISTANCE_ATR,
    slippage_bps=0.0,
):

    direction = event["direction"]
    event_pos = int(event["bar_pos"])
    event_time = event["event_time"]

    origin = float(event["origin_price"])
    atr = float(event["atr"])

    if np.isnan(atr) or atr <= 0:
        return None

    running_extreme = float(event["event_extreme"])

    armed = False
    armed_pos = None
    limit_price = None

    next_choch = supersession_info.get("next_choch")
    next_opposite_bos = supersession_info.get("next_opposite_bos")

    def displacement(extreme):
        if direction == "bullish":
            return (extreme - origin) / atr
        return (origin - extreme) / atr

    def fib(extreme):
        return retracement_price(
            origin,
            extreme,
            direction,
            retrace,
        )

    # Order can only become active after CHoCH candle closes.
    if displacement(running_extreme) >= min_displacement_atr:
        armed = True
        armed_pos = event_pos
        limit_price = fib(running_extreme)

    search_end = min(
        len(df),
        event_pos + 1 + pending_expiry_bars + 100
    )

    for pos in range(event_pos + 1, search_end):

        t = df.index[pos]
        bar = df.iloc[pos]

        # New structure supersedes pending setup.
        if (
            CANCEL_ON_NEW_CHOCH
            and next_choch is not None
            and t >= next_choch
        ):
            return {
                "status":"cancel_new_choch",
                "event_time":event_time,
                "armed":armed,
                "filled":False,
            }

        if (
            CANCEL_ON_OPPOSITE_BOS
            and next_opposite_bos is not None
            and t >= next_opposite_bos
        ):
            return {
                "status":"cancel_opposite_bos",
                "event_time":event_time,
                "armed":armed,
                "filled":False,
            }

        # Existing limit is live during this bar.
        if armed and limit_price is not None:

            if pos - armed_pos > pending_expiry_bars:
                return {
                    "status":"cancel_expired",
                    "event_time":event_time,
                    "armed":True,
                    "filled":False,
                }

            if pending_limit_filled(
                bar,
                direction,
                limit_price,
                asset_spec["fallback_spread_bps"],
            ):

                frozen_extreme = running_extreme

                slip = slippage_bps / 10000.0

                entry_price = (
                    limit_price
                    * (
                        1 + slip
                        if direction == "bullish"
                        else 1 - slip
                    )
                )

                stop = origin
                target = frozen_extreme

                risk = abs(entry_price - stop)

                if risk <= 0:
                    return None

                future = df.iloc[
                    pos:
                    min(len(df), pos + MAX_HOLD_BARS)
                ]

                outcome = None
                exit_price = None
                exit_time = None
                exit_fee_bps = asset_spec["taker_bps"]

                for exit_t, exit_bar in future.iterrows():

                    side = executable_side(
                        exit_bar,
                        direction
                    )

                    if direction == "bullish":
                        sl_hit = side["low"] <= stop
                        tp_hit = side["high"] >= target
                    else:
                        sl_hit = side["high"] >= stop
                        tp_hit = side["low"] <= target

                    if sl_hit and tp_hit:
                        outcome = (
                            "sl"
                            if INTRABAR_PRIORITY == "stop_first"
                            else "tp"
                        )
                    elif sl_hit:
                        outcome = "sl"
                    elif tp_hit:
                        outcome = "tp"
                    else:
                        continue

                    exit_time = exit_t

                    if outcome == "sl":
                        exit_price = (
                            stop
                            * (
                                1 - slip
                                if direction == "bullish"
                                else 1 + slip
                            )
                        )
                        exit_fee_bps = asset_spec["taker_bps"]
                    else:
                        exit_price = (
                            target
                            * (
                                1 - slip
                                if direction == "bullish"
                                else 1 + slip
                            )
                        )
                        exit_fee_bps = asset_spec["maker_bps"]

                    break

                if outcome is None:

                    last = future.iloc[-1]
                    exit_time = future.index[-1]

                    if direction == "bullish" and "bidclose" in future.columns:
                        exit_price = float(last["bidclose"])
                    elif direction == "bearish" and "askclose" in future.columns:
                        exit_price = float(last["askclose"])
                    else:
                        exit_price = float(last["close"])

                    outcome = "timeout"
                    exit_fee_bps = asset_spec["taker_bps"]

                gross_r = (
                    (exit_price - entry_price) / risk
                    if direction == "bullish"
                    else (entry_price - exit_price) / risk
                )

                fees_r = fee_in_r(
                    entry_price,
                    exit_price,
                    risk,
                    asset_spec["maker_bps"],
                    exit_fee_bps,
                )

                net_r = gross_r - fees_r

                return {
                    "status":"filled",
                    "event_time":event_time,
                    "entry_time":t,
                    "exit_time":exit_time,
                    "armed":True,
                    "filled":True,
                    "entry":entry_price,
                    "stop":stop,
                    "target":target,
                    "risk_price":risk,
                    "frozen_extreme":frozen_extreme,
                    "gross_r":gross_r,
                    "fees_r":fees_r,
                    "net_r":net_r,
                    "winner":bool(net_r > 0),
                    "outcome":outcome,
                }

        # Origin broken before fill.
        if INVALIDATE_ON_ORIGIN_BREAK:

            if direction == "bullish" and float(bar["low"]) <= origin:
                return {
                    "status":"cancel_origin_break",
                    "event_time":event_time,
                    "armed":armed,
                    "filled":False,
                }

            if direction == "bearish" and float(bar["high"]) >= origin:
                return {
                    "status":"cancel_origin_break",
                    "event_time":event_time,
                    "armed":armed,
                    "filled":False,
                }

        # Candle CLOSE: update impulse extreme.
        new_extreme = False

        if direction == "bullish":

            e = float(bar["high"])

            if e > running_extreme:
                running_extreme = e
                new_extreme = True

        else:

            e = float(bar["low"])

            if e < running_extreme:
                running_extreme = e
                new_extreme = True

        current_disp = displacement(
            running_extreme
        )

        if (
            not armed
            and current_disp >= min_displacement_atr
        ):
            armed = True
            armed_pos = pos
            limit_price = fib(running_extreme)

        elif armed and new_extreme:
            # Cancel/replace only after candle closes.
            limit_price = fib(running_extreme)

        if armed and limit_price is not None:

            distance_atr = (
                abs(float(bar["close"]) - limit_price)
                / atr
            )

            if distance_atr > max_entry_distance_atr:
                return {
                    "status":"cancel_too_far",
                    "event_time":event_time,
                    "armed":True,
                    "filled":False,
                }

    return {
        "status":"unresolved",
        "event_time":event_time,
        "armed":armed,
        "filled":False,
    }

def summarize_dynamic(df):

    if df is None or df.empty:
        return {
            "events":0,
            "armed":0,
            "filled":0,
            "fill_rate":np.nan,
            "avg_r":np.nan,
            "median_r":np.nan,
            "win_rate":np.nan,
            "total_r":0.0,
        }

    events = len(df)
    armed = int(df["armed"].fillna(False).sum())

    filled = df[
        df["filled"].eq(True)
    ].copy()

    if filled.empty:
        return {
            "events":events,
            "armed":armed,
            "filled":0,
            "fill_rate":0.0,
            "avg_r":np.nan,
            "median_r":np.nan,
            "win_rate":np.nan,
            "total_r":0.0,
        }

    return {
        "events":events,
        "armed":armed,
        "filled":len(filled),
        "fill_rate":len(filled)/events,
        "avg_r":filled["net_r"].mean(),
        "median_r":filled["net_r"].median(),
        "win_rate":filled["winner"].eq(True).mean(),
        "total_r":filled["net_r"].sum(),
    }

print("Dynamic simulator loaded.")


# ============================================================
# CELL 7 — BUILD FROZEN ENTRY SETUPS
# ============================================================
# We first identify the exact same frozen signal entries.
# Exit policy is NOT involved in setup selection.
# ============================================================

def collect_frozen_entries():
    rows = []

    for pair, df in DATA.items():
        spec = ASSETS[pair]
        events = EVENTS[pair]

        choch = events[
            (events["type"] == "CHoCH")
            & (events["broken_structure_atr"] >= FROZEN_MIN_BROKEN_STRUCTURE_ATR)
            & (events["break_strength_atr"] >= FROZEN_MIN_CLOSE_THROUGH_ATR)
        ].copy()

        supers = SUPERSESSION[pair]

        for _, event in choch.iterrows():
            # Use the existing simulator only to establish the causal fill.
            # We ignore its eventual exit and rebuild exit logic later.
            res = simulate_dynamic_trade(
                df=df,
                event=event,
                asset_spec=spec,
                supersession_info=supers.get(int(event["bar_pos"]), {}),
                min_displacement_atr=FROZEN_MIN_DISPLACEMENT_ATR,
                retrace=RETRACE,
                pending_expiry_bars=PENDING_EXPIRY_BARS,
                max_entry_distance_atr=MAX_ENTRY_DISTANCE_ATR,
                slippage_bps=0.0,
            )

            if res is None or not res.get("filled", False):
                continue

            # Reconstruct entry risk from actual entry + structural origin.
            entry_time = pd.Timestamp(res["entry_time"])
            entry_pos = df.index.get_indexer([entry_time])[0]

            entry_price = float(res["entry"])
            stop_price = float(event["origin_price"])
            risk = abs(entry_price - stop_price)

            if risk <= 0:
                continue

            structural_target = float(res["target"])

            rows.append({
                "pair": pair,
                "event_time": pd.Timestamp(event["event_time"]),
                "entry_time": entry_time,
                "entry_pos": int(entry_pos),
                "direction": event["direction"],
                "entry_price": entry_price,
                "stop_price": stop_price,
                "risk_price": risk,
                "structural_target": structural_target,
            })

    return pd.DataFrame(rows).sort_values(
        ["entry_time", "pair"]
    ).reset_index(drop=True)

entries_df = collect_frozen_entries()

print("Frozen entries:", len(entries_df))




# ============================================================
# STRATEGY 2 — CONSTRAINED EVOLUTIONARY EXECUTION SEARCH
# ============================================================
#
# Research question:
# After frozen Strategy 1 reaches +1.5R, can a SIMPLE causal rule
# on 5m / 10m / 15m identify exhaustion early enough to trade
# the retracement robustly?
#
# Search discipline:
#   - Frozen Strategy 1 is NOT changed.
#   - Discovery fitness uses 2022-2024 only.
#   - 2025 is used as a chronological validation year.
#   - 2026 is revealed only after finalists are selected.
#   - Complexity and missing-year / missing-market solutions are penalised.
#   - We inspect the top population, not only one lucky genome.
#
# NOTE:
# 2025/2026 have already been viewed in earlier research, so they are
# chronological validation checks rather than pristine unseen OOS.
# Future paper/live data remains the true untouched holdout.
# ============================================================

FROZEN_TP_R = 1.5

SEARCH_TF_MINUTES = [5, 10, 15]
SEARCH_MAX_DECISION_MINUTES = 120

DISCOVERY_YEARS = [2022, 2023, 2024]
VALIDATION_YEAR = 2025
HOLDOUT_YEAR = 2026

GA_SEED = 42
GA_POPULATION = 64
GA_GENERATIONS = 28
GA_ELITE = 12
GA_RANDOM_INJECT = 10

MIN_DISCOVERY_TRADES = 24
MIN_YEAR_TRADES = 4
MIN_MARKET_TRADES = 6

# Keep execution costs represented by native bid/ask where QuoteBars exist.
# Extra slippage is deliberately zero during the search. Stress it only
# after a stable candidate is found.
EXTRA_SLIPPAGE_BPS = 0.0

rng = np.random.default_rng(GA_SEED)


# ============================================================
# A. FIXED 1.5R STRATEGY-1 TP WINNERS
# ============================================================

def s1_price_at_r(entry, risk, direction, r_value):
    if direction == "bullish":
        return entry + r_value * risk
    return entry - r_value * risk


def locate_frozen_tp(df, row):
    entry_pos = int(row["entry_pos"])
    entry = float(row["entry_price"])
    stop = float(row["stop_price"])
    risk = float(row["risk_price"])
    direction = row["direction"]

    tp = s1_price_at_r(entry, risk, direction, FROZEN_TP_R)

    future = df.iloc[
        entry_pos:
        min(len(df), entry_pos + MAX_HOLD_BARS)
    ]

    if future.empty:
        return None

    for t, bar in future.iterrows():
        side = executable_side(bar, direction)

        if direction == "bullish":
            stop_hit = side["low"] <= stop
            tp_hit = side["high"] >= tp
        else:
            stop_hit = side["high"] >= stop
            tp_hit = side["low"] <= tp

        # Same-bar ambiguity remains conservative: stop first.
        if stop_hit:
            return None

        if tp_hit:
            return pd.Timestamp(t)

    return None


tp_rows = []

for _, row in entries_df.iterrows():
    pair = row["pair"]
    tp_time = locate_frozen_tp(DATA[pair], row)

    if tp_time is None:
        continue

    tp_rows.append({
        "pair": pair,
        "entry_time": pd.Timestamp(row["entry_time"]),
        "tp_time": tp_time,
        "year": tp_time.year,
        "direction": row["direction"],
        "original_entry": float(row["entry_price"]),
        "original_risk": float(row["risk_price"]),
    })

tp_winners_df = (
    pd.DataFrame(tp_rows)
    .sort_values(["tp_time", "pair"])
    .reset_index(drop=True)
)

print("Frozen Strategy-1 +1.5R winners:", len(tp_winners_df))
print(tp_winners_df.groupby(["year", "pair"]).size())


# ============================================================
# B. BUILD 5m / 10m / 15m DATA + ATR
# ============================================================

def add_search_atr(df, period=14):
    x = df.copy()
    prev = x["close"].shift(1)

    tr = pd.concat(
        [
            x["high"] - x["low"],
            (x["high"] - prev).abs(),
            (x["low"] - prev).abs(),
        ],
        axis=1,
    ).max(axis=1)

    x["atr_s2"] = tr.rolling(
        period,
        min_periods=period,
    ).mean()

    return x


LOWER_DATA = {}

for tf in SEARCH_TF_MINUTES:
    LOWER_DATA[tf] = {}

    for pair, df1m in DATA_1M.items():
        d = resample_ohlc(df1m, tf)
        d = add_search_atr(d, ATR_PERIOD)
        LOWER_DATA[tf][pair] = d

        print(
            f"{tf}m {pair}: {len(d):,} bars"
        )


# ============================================================
# C. CAUSAL FEATURE CACHE
# ============================================================

def original_exec_ohlc(bar, original_direction):
    # Observe the path from Strategy-1's executable side.
    if original_direction == "bullish" and "bidopen" in bar.index:
        return (
            float(bar["bidopen"]),
            float(bar["bidhigh"]),
            float(bar["bidlow"]),
            float(bar["bidclose"]),
        )

    if original_direction == "bearish" and "askopen" in bar.index:
        return (
            float(bar["askopen"]),
            float(bar["askhigh"]),
            float(bar["asklow"]),
            float(bar["askclose"]),
        )

    return (
        float(bar["open"]),
        float(bar["high"]),
        float(bar["low"]),
        float(bar["close"]),
    )


def price_to_original_r(price, entry, risk, direction):
    if direction == "bullish":
        return (price - entry) / risk
    return (entry - price) / risk


feature_rows = []

for tp_id, row in tp_winners_df.iterrows():
    pair = row["pair"]
    tp_time = pd.Timestamp(row["tp_time"])
    direction = row["direction"]
    original_entry = float(row["original_entry"])
    original_risk = float(row["original_risk"])

    for tf in SEARCH_TF_MINUTES:
        df = LOWER_DATA[tf][pair]

        window = df[df.index > tp_time].iloc[
            : int(SEARCH_MAX_DECISION_MINUTES / tf)
        ]

        if window.empty:
            continue

        lost_1p5 = False
        reclaimed_after_loss = False
        against_run = 0
        max_against_run = 0
        against_count = 0

        rolling_adverse_prices = []

        for bar_num, (signal_time, bar) in enumerate(
            window.iterrows(),
            start=1,
        ):
            if pd.isna(bar["atr_s2"]):
                continue

            o, h, l, c = original_exec_ohlc(
                bar,
                direction,
            )

            open_r = price_to_original_r(
                o, original_entry, original_risk, direction
            )
            close_r = price_to_original_r(
                c, original_entry, original_risk, direction
            )

            closes_against = close_r < open_r

            if closes_against:
                against_count += 1
                against_run += 1
                max_against_run = max(max_against_run, against_run)
            else:
                against_run = 0

            if close_r < FROZEN_TP_R:
                lost_1p5 = True
            elif lost_1p5 and close_r >= FROZEN_TP_R:
                reclaimed_after_loss = True

            # Raw adverse extreme for the eventual reversal stop.
            if direction == "bullish":
                # Strategy 2 will be short, so adverse = high.
                adverse_price = h
            else:
                # Strategy 2 will be long, so adverse = low.
                adverse_price = l

            rolling_adverse_prices.append(adverse_price)

            feature_rows.append({
                "tp_id": int(tp_id),
                "pair": pair,
                "year": int(row["year"]),
                "tf": int(tf),
                "bar_num": int(bar_num),
                "decision_minutes": int(bar_num * tf),
                "signal_time": pd.Timestamp(signal_time),
                "direction": direction,
                "signal_close": float(c),
                "signal_high": float(h),
                "signal_low": float(l),
                "atr": float(bar["atr_s2"]),
                "close_r": float(close_r),
                "against_ratio": float(against_count / bar_num),
                "max_against_run": int(max_against_run),
                "lost_1p5": bool(lost_1p5),
                "reclaimed_after_loss": bool(reclaimed_after_loss),
                "roll_adverse_2": float(
                    max(rolling_adverse_prices[-2:])
                    if direction == "bullish"
                    else min(rolling_adverse_prices[-2:])
                ),
                "roll_adverse_3": float(
                    max(rolling_adverse_prices[-3:])
                    if direction == "bullish"
                    else min(rolling_adverse_prices[-3:])
                ),
                "roll_adverse_4": float(
                    max(rolling_adverse_prices[-4:])
                    if direction == "bullish"
                    else min(rolling_adverse_prices[-4:])
                ),
            })

features_df = pd.DataFrame(feature_rows)

print("Cached decision states:", len(features_df))


# ============================================================
# D. GENOME
# ============================================================

STOP_TYPES = ["signal_extreme", "rolling_extreme", "atr"]
TF_CHOICES = SEARCH_TF_MINUTES


def random_genome():
    tf = int(rng.choice(TF_CHOICES))
    max_bar = max(1, int(SEARCH_MAX_DECISION_MINUTES / tf))

    g = {
        "tf": tf,
        "decision_bar": int(rng.integers(1, max_bar + 1)),

        # Entry conditions.
        "use_close": bool(rng.integers(0, 2)),
        "close_threshold": float(rng.uniform(0.45, 1.60)),

        "use_against_ratio": bool(rng.integers(0, 2)),
        "against_ratio": float(rng.uniform(0.25, 0.90)),

        "use_consecutive": bool(rng.integers(0, 2)),
        "min_consecutive": int(rng.integers(1, 5)),

        "require_lost_1p5": bool(rng.integers(0, 2)),
        "require_no_reclaim": bool(rng.integers(0, 2)),

        # Strategy-2 risk geometry.
        "stop_type": str(rng.choice(STOP_TYPES)),
        "stop_lookback": int(rng.integers(2, 5)),
        "atr_buffer": float(rng.uniform(0.00, 0.40)),
        "atr_stop_mult": float(rng.uniform(0.60, 2.20)),

        "target_r": float(rng.uniform(0.75, 3.00)),
        "max_hold_minutes": int(rng.choice(
            np.arange(60, 721, 30)
        )),
    }

    # Never allow an empty entry rule.
    if not (
        g["use_close"]
        or g["use_against_ratio"]
        or g["use_consecutive"]
        or g["require_lost_1p5"]
    ):
        g["use_close"] = True

    # "No reclaim" only makes sense if the 1.5R area had first been lost.
    if g["require_no_reclaim"]:
        g["require_lost_1p5"] = True

    return g


def repair_genome(g):
    g = dict(g)

    g["tf"] = int(min(
        TF_CHOICES,
        key=lambda x: abs(x - int(g["tf"]))
    ))

    max_bar = max(1, int(SEARCH_MAX_DECISION_MINUTES / g["tf"]))
    g["decision_bar"] = int(np.clip(g["decision_bar"], 1, max_bar))

    g["close_threshold"] = float(np.clip(g["close_threshold"], 0.45, 1.60))
    g["against_ratio"] = float(np.clip(g["against_ratio"], 0.25, 0.90))
    g["min_consecutive"] = int(np.clip(g["min_consecutive"], 1, 4))
    g["stop_lookback"] = int(np.clip(g["stop_lookback"], 2, 4))
    g["atr_buffer"] = float(np.clip(g["atr_buffer"], 0.00, 0.40))
    g["atr_stop_mult"] = float(np.clip(g["atr_stop_mult"], 0.60, 2.20))
    g["target_r"] = float(np.clip(g["target_r"], 0.75, 3.00))

    hold_choices = np.arange(60, 721, 30)
    g["max_hold_minutes"] = int(min(
        hold_choices,
        key=lambda x: abs(x - int(g["max_hold_minutes"]))
    ))

    if g["stop_type"] not in STOP_TYPES:
        g["stop_type"] = "signal_extreme"

    if not (
        g["use_close"]
        or g["use_against_ratio"]
        or g["use_consecutive"]
        or g["require_lost_1p5"]
    ):
        g["use_close"] = True

    if g["require_no_reclaim"]:
        g["require_lost_1p5"] = True

    return g


def genome_complexity(g):
    conditions = (
        int(g["use_close"])
        + int(g["use_against_ratio"])
        + int(g["use_consecutive"])
        + int(g["require_lost_1p5"])
        + int(g["require_no_reclaim"])
    )

    # Slight extra cost for more bespoke stop logic.
    stop_cost = (
        0.0
        if g["stop_type"] == "signal_extreme"
        else 0.5
    )

    return conditions + stop_cost


# ============================================================
# E. SIGNAL FILTER + CAUSAL TRADE SIMULATOR
# ============================================================

def select_genome_signals(g, years):
    x = features_df[
        (features_df["tf"] == g["tf"])
        & (features_df["bar_num"] == g["decision_bar"])
        & (features_df["year"].isin(years))
    ].copy()

    if x.empty:
        return x

    mask = pd.Series(True, index=x.index)

    if g["use_close"]:
        mask &= x["close_r"] <= g["close_threshold"]

    if g["use_against_ratio"]:
        mask &= x["against_ratio"] >= g["against_ratio"]

    if g["use_consecutive"]:
        mask &= x["max_against_run"] >= g["min_consecutive"]

    if g["require_lost_1p5"]:
        mask &= x["lost_1p5"]

    if g["require_no_reclaim"]:
        mask &= ~x["reclaimed_after_loss"]

    return x[mask]


def simulate_reversal_from_feature(r, g):
    tf = int(g["tf"])
    df = LOWER_DATA[tf][r["pair"]]

    signal_time = pd.Timestamp(r["signal_time"])
    pos = df.index.get_indexer([signal_time])[0]

    if pos < 0:
        return None

    entry = float(r["signal_close"])
    atr = float(r["atr"])
    direction = r["direction"]

    if not np.isfinite(atr) or atr <= 0:
        return None

    # Strategy 2 is opposite Strategy 1.
    if direction == "bullish":
        # reversal short
        if g["stop_type"] == "signal_extreme":
            stop = float(r["signal_high"]) + g["atr_buffer"] * atr
        elif g["stop_type"] == "rolling_extreme":
            stop = float(r[f"roll_adverse_{g['stop_lookback']}"]) + g["atr_buffer"] * atr
        else:
            stop = entry + g["atr_stop_mult"] * atr

        risk = stop - entry
        if risk <= 0:
            return None

        target = entry - g["target_r"] * risk

    else:
        # reversal long
        if g["stop_type"] == "signal_extreme":
            stop = float(r["signal_low"]) - g["atr_buffer"] * atr
        elif g["stop_type"] == "rolling_extreme":
            stop = float(r[f"roll_adverse_{g['stop_lookback']}"]) - g["atr_buffer"] * atr
        else:
            stop = entry - g["atr_stop_mult"] * atr

        risk = entry - stop
        if risk <= 0:
            return None

        target = entry + g["target_r"] * risk

    max_hold_bars = max(
        1,
        int(np.ceil(g["max_hold_minutes"] / tf))
    )

    future = df.iloc[
        pos + 1:
        min(len(df), pos + 1 + max_hold_bars)
    ]

    if future.empty:
        return None

    slip = EXTRA_SLIPPAGE_BPS / 10000.0

    for t, bar in future.iterrows():
        if direction == "bullish":
            # reversal short exits on ask
            if "askhigh" in bar.index:
                high = float(bar["askhigh"])
                low = float(bar["asklow"])
            else:
                high = float(bar["high"])
                low = float(bar["low"])

            stop_hit = high >= stop
            target_hit = low <= target

        else:
            # reversal long exits on bid
            if "bidhigh" in bar.index:
                high = float(bar["bidhigh"])
                low = float(bar["bidlow"])
            else:
                high = float(bar["high"])
                low = float(bar["low"])

            stop_hit = low <= stop
            target_hit = high >= target

        # Conservative same-bar ambiguity.
        if stop_hit:
            return {
                "r": -1.0,
                "win": False,
                "outcome": "stop",
                "exit_time": pd.Timestamp(t),
            }

        if target_hit:
            return {
                "r": float(g["target_r"]),
                "win": True,
                "outcome": "target",
                "exit_time": pd.Timestamp(t),
            }

    last = future.iloc[-1]

    if direction == "bullish":
        close = (
            float(last["askclose"])
            if "askclose" in last.index
            else float(last["close"])
        )
        pnl = entry - close
    else:
        close = (
            float(last["bidclose"])
            if "bidclose" in last.index
            else float(last["close"])
        )
        pnl = close - entry

    return {
        "r": float(pnl / risk),
        "win": bool(pnl > 0),
        "outcome": "timeout",
        "exit_time": pd.Timestamp(future.index[-1]),
    }


def evaluate_genome(g, years):
    signals = select_genome_signals(g, years)

    if signals.empty:
        return pd.DataFrame()

    rows = []

    for _, r in signals.iterrows():
        res = simulate_reversal_from_feature(r, g)

        if res is None:
            continue

        rows.append({
            "tp_id": int(r["tp_id"]),
            "pair": r["pair"],
            "year": int(r["year"]),
            "signal_time": pd.Timestamp(r["signal_time"]),
            **res,
        })

    return pd.DataFrame(rows)


def max_drawdown_r(trades):
    if trades is None or trades.empty:
        return np.nan

    x = (
        trades
        .sort_values(["signal_time", "pair"])["r"]
        .cumsum()
        .to_numpy(float)
    )

    peaks = np.maximum.accumulate(
        np.concatenate([[0.0], x])
    )[1:]

    dd = x - peaks

    return float(dd.min()) if len(dd) else 0.0


# ============================================================
# F. DISCOVERY FITNESS — 2022-2024 ONLY
# ============================================================

fitness_cache = {}


def genome_key(g):
    # Rounded key keeps the cache useful after mutation.
    return (
        int(g["tf"]),
        int(g["decision_bar"]),
        bool(g["use_close"]),
        round(float(g["close_threshold"]), 3),
        bool(g["use_against_ratio"]),
        round(float(g["against_ratio"]), 3),
        bool(g["use_consecutive"]),
        int(g["min_consecutive"]),
        bool(g["require_lost_1p5"]),
        bool(g["require_no_reclaim"]),
        g["stop_type"],
        int(g["stop_lookback"]),
        round(float(g["atr_buffer"]), 3),
        round(float(g["atr_stop_mult"]), 3),
        round(float(g["target_r"]), 3),
        int(g["max_hold_minutes"]),
    )


def discovery_fitness(g):
    g = repair_genome(g)
    key = genome_key(g)

    if key in fitness_cache:
        return fitness_cache[key]

    trades = evaluate_genome(g, DISCOVERY_YEARS)

    if trades.empty:
        result = (-999.0, None)
        fitness_cache[key] = result
        return result

    n = len(trades)

    yearly = (
        trades
        .groupby("year")
        .agg(
            trades=("r", "size"),
            avg_r=("r", "mean"),
            total_r=("r", "sum"),
        )
    )

    markets = (
        trades
        .groupby("pair")
        .agg(
            trades=("r", "size"),
            avg_r=("r", "mean"),
            total_r=("r", "sum"),
        )
    )

    # Hard-ish penalties: a strategy must exist across all discovery
    # years and both frozen markets to score well.
    missing_years = len(set(DISCOVERY_YEARS) - set(yearly.index))
    missing_markets = len(set(ASSETS) - set(markets.index))

    low_year_count = int(
        (yearly["trades"] < MIN_YEAR_TRADES).sum()
    )

    low_market_count = int(
        (markets["trades"] < MIN_MARKET_TRADES).sum()
    )

    overall_avg = float(trades["r"].mean())
    median_year_avg = float(yearly["avg_r"].median())
    worst_year_avg = float(yearly["avg_r"].min())
    mean_market_avg = float(markets["avg_r"].mean())
    worst_market_avg = float(markets["avg_r"].min())
    dd = max_drawdown_r(trades)
    complexity = genome_complexity(g)

    positive_year_fraction = float(
        (yearly["avg_r"] > 0).mean()
    )

    positive_market_fraction = float(
        (markets["avg_r"] > 0).mean()
    )

    # Stability first, raw expectancy second.
    score = (
        0.34 * median_year_avg
        + 0.24 * worst_year_avg
        + 0.16 * overall_avg
        + 0.12 * mean_market_avg
        + 0.08 * worst_market_avg
        + 0.04 * positive_year_fraction
        + 0.02 * positive_market_fraction
        - 0.010 * complexity
        - 0.003 * abs(min(dd, 0.0))
    )

    if n < MIN_DISCOVERY_TRADES:
        score -= 0.35 * (
            (MIN_DISCOVERY_TRADES - n)
            / MIN_DISCOVERY_TRADES
        )

    score -= 0.60 * missing_years
    score -= 0.75 * missing_markets
    score -= 0.20 * low_year_count
    score -= 0.20 * low_market_count

    metrics = {
        "trades": int(n),
        "avg_r": overall_avg,
        "median_year_avg": median_year_avg,
        "worst_year_avg": worst_year_avg,
        "positive_years": int((yearly["avg_r"] > 0).sum()),
        "years": int(len(yearly)),
        "positive_markets": int((markets["avg_r"] > 0).sum()),
        "markets": int(len(markets)),
        "worst_market_avg": worst_market_avg,
        "max_dd_r": float(dd),
        "complexity": float(complexity),
    }

    result = (float(score), metrics)
    fitness_cache[key] = result
    return result


# ============================================================
# G. EVOLUTIONARY OPERATORS
# ============================================================

BOOL_FIELDS = [
    "use_close",
    "use_against_ratio",
    "use_consecutive",
    "require_lost_1p5",
    "require_no_reclaim",
]

FLOAT_FIELDS = [
    "close_threshold",
    "against_ratio",
    "atr_buffer",
    "atr_stop_mult",
    "target_r",
]


def crossover(a, b):
    child = {}

    for k in a:
        child[k] = copy.deepcopy(
            a[k] if rng.random() < 0.5 else b[k]
        )

    return repair_genome(child)


def mutate(g, rate=0.22):
    g = dict(g)

    if rng.random() < rate:
        g["tf"] = int(rng.choice(TF_CHOICES))

    if rng.random() < rate:
        max_bar = max(1, int(SEARCH_MAX_DECISION_MINUTES / g["tf"]))
        g["decision_bar"] = int(
            np.clip(
                g["decision_bar"] + rng.integers(-3, 4),
                1,
                max_bar,
            )
        )

    for k in BOOL_FIELDS:
        if rng.random() < rate * 0.65:
            g[k] = not bool(g[k])

    if rng.random() < rate:
        g["close_threshold"] += float(rng.normal(0, 0.12))

    if rng.random() < rate:
        g["against_ratio"] += float(rng.normal(0, 0.08))

    if rng.random() < rate:
        g["min_consecutive"] += int(rng.choice([-1, 1]))

    if rng.random() < rate:
        g["stop_type"] = str(rng.choice(STOP_TYPES))

    if rng.random() < rate:
        g["stop_lookback"] += int(rng.choice([-1, 1]))

    if rng.random() < rate:
        g["atr_buffer"] += float(rng.normal(0, 0.06))

    if rng.random() < rate:
        g["atr_stop_mult"] += float(rng.normal(0, 0.20))

    if rng.random() < rate:
        g["target_r"] += float(rng.normal(0, 0.20))

    if rng.random() < rate:
        g["max_hold_minutes"] += int(rng.choice([-60, -30, 30, 60]))

    return repair_genome(g)


def tournament(population, scored, k=4):
    idxs = rng.choice(
        len(population),
        size=min(k, len(population)),
        replace=False,
    )

    best_i = max(
        idxs,
        key=lambda i: scored[i][0],
    )

    return population[best_i]


# ============================================================
# H. RUN EVOLUTION
# ============================================================

population = [
    random_genome()
    for _ in range(GA_POPULATION)
]

generation_rows = []

for generation in range(GA_GENERATIONS):
    scored = [
        discovery_fitness(g)
        for g in population
    ]

    order = np.argsort(
        [x[0] for x in scored]
    )[::-1]

    population = [
        population[i]
        for i in order
    ]

    scored = [
        scored[i]
        for i in order
    ]

    best_score = scored[0][0]
    best_metrics = scored[0][1] or {}

    generation_rows.append({
        "generation": generation,
        "best_score": best_score,
        "best_avg_r": best_metrics.get("avg_r", np.nan),
        "best_trades": best_metrics.get("trades", 0),
        "cache_size": len(fitness_cache),
    })

    print(
        f"GA {generation+1:02d}/{GA_GENERATIONS} | "
        f"score={best_score:.4f} | "
        f"avgR={best_metrics.get('avg_r', np.nan):.4f} | "
        f"n={best_metrics.get('trades', 0)} | "
        f"cache={len(fitness_cache):,}"
    )

    elites = [
        copy.deepcopy(g)
        for g in population[:GA_ELITE]
    ]

    next_population = elites[:]

    while len(next_population) < (
        GA_POPULATION - GA_RANDOM_INJECT
    ):
        p1 = tournament(population, scored)
        p2 = tournament(population, scored)

        child = crossover(p1, p2)
        child = mutate(child)

        next_population.append(child)

    while len(next_population) < GA_POPULATION:
        next_population.append(
            random_genome()
        )

    population = next_population

ga_history_df = pd.DataFrame(generation_rows)


# ============================================================
# I. COLLECT UNIQUE TOP DISCOVERY SOLUTIONS
# ============================================================

# Combine final population + every cached genome represented by the
# final population variants. Then de-duplicate by rounded genome key.
final_scored = []

for g in population:
    score, metrics = discovery_fitness(g)

    if metrics is None:
        continue

    final_scored.append((
        score,
        repair_genome(g),
        metrics,
    ))

# Add random local neighbours around the current top population so we
# inspect a basin rather than a single point.
top_seed = sorted(
    final_scored,
    key=lambda x: x[0],
    reverse=True,
)[:20]

for _, g, _ in top_seed:
    for _ in range(8):
        ng = mutate(g, rate=0.35)
        score, metrics = discovery_fitness(ng)

        if metrics is not None:
            final_scored.append((
                score,
                ng,
                metrics,
            ))

unique = {}

for score, g, metrics in final_scored:
    key = genome_key(g)

    if (
        key not in unique
        or score > unique[key][0]
    ):
        unique[key] = (
            score,
            g,
            metrics,
        )

ranked_discovery = sorted(
    unique.values(),
    key=lambda x: x[0],
    reverse=True,
)

discovery_rows = []

for rank, (score, g, m) in enumerate(
    ranked_discovery[:60],
    start=1,
):
    discovery_rows.append({
        "rank": rank,
        "score": score,
        **g,
        **m,
        "decision_minutes": g["tf"] * g["decision_bar"],
    })

discovery_top_df = pd.DataFrame(discovery_rows)


# ============================================================
# J. 2025 CHRONOLOGICAL VALIDATION OF TOP DISCOVERY CANDIDATES
# ============================================================

validation_rows = []

for _, row in discovery_top_df.head(40).iterrows():
    g = {
        k: row[k]
        for k in [
            "tf",
            "decision_bar",
            "use_close",
            "close_threshold",
            "use_against_ratio",
            "against_ratio",
            "use_consecutive",
            "min_consecutive",
            "require_lost_1p5",
            "require_no_reclaim",
            "stop_type",
            "stop_lookback",
            "atr_buffer",
            "atr_stop_mult",
            "target_r",
            "max_hold_minutes",
        ]
    }

    g = repair_genome(g)

    val = evaluate_genome(
        g,
        [VALIDATION_YEAR],
    )

    if val.empty:
        val_n = 0
        val_avg = np.nan
        val_total = np.nan
        val_dd = np.nan
        val_market_pos = 0
    else:
        val_n = len(val)
        val_avg = float(val["r"].mean())
        val_total = float(val["r"].sum())
        val_dd = max_drawdown_r(val)

        vm = val.groupby("pair")["r"].mean()
        val_market_pos = int((vm > 0).sum())

    # Validation-weighted score, but do not reward a 1-trade miracle.
    val_score = (
        float(row["score"])
        + (0.40 * val_avg if np.isfinite(val_avg) else -0.75)
        + 0.05 * val_market_pos
        - (
            0.30 * max(0, MIN_YEAR_TRADES - val_n) / MIN_YEAR_TRADES
            if val_n < MIN_YEAR_TRADES
            else 0.0
        )
    )

    validation_rows.append({
        **row.to_dict(),
        "val_2025_trades": val_n,
        "val_2025_avg_r": val_avg,
        "val_2025_total_r": val_total,
        "val_2025_dd_r": val_dd,
        "val_2025_positive_markets": val_market_pos,
        "validation_score": val_score,
    })

validated_df = (
    pd.DataFrame(validation_rows)
    .sort_values(
        [
            "validation_score",
            "val_2025_avg_r",
            "score",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

# A provisional finalist must at least have:
# - enough discovery trades
# - all 3 discovery years represented
# - both markets represented
# - positive discovery avgR
# - non-negative 2025 validation avgR
eligible_finalists = validated_df[
    (validated_df["trades"] >= MIN_DISCOVERY_TRADES)
    & (validated_df["years"] == len(DISCOVERY_YEARS))
    & (validated_df["markets"] == len(ASSETS))
    & (validated_df["avg_r"] > 0)
    & (validated_df["val_2025_trades"] >= MIN_YEAR_TRADES)
    & (validated_df["val_2025_avg_r"] >= 0)
].copy()

if eligible_finalists.empty:
    provisional_winner = None
    holdout_df = pd.DataFrame()
    combined_detail_df = pd.DataFrame()
else:
    provisional_winner = eligible_finalists.iloc[0]

    WINNER_FIELDS = [
        "tf",
        "decision_bar",
        "use_close",
        "close_threshold",
        "use_against_ratio",
        "against_ratio",
        "use_consecutive",
        "min_consecutive",
        "require_lost_1p5",
        "require_no_reclaim",
        "stop_type",
        "stop_lookback",
        "atr_buffer",
        "atr_stop_mult",
        "target_r",
        "max_hold_minutes",
    ]

    winner_genome = repair_genome({
        k: provisional_winner[k]
        for k in WINNER_FIELDS
    })

    holdout_df = evaluate_genome(
        winner_genome,
        [HOLDOUT_YEAR],
    )

    combined_detail_df = evaluate_genome(
        winner_genome,
        DISCOVERY_YEARS + [VALIDATION_YEAR, HOLDOUT_YEAR],
    )


# ============================================================
# K. FINAL DIAGNOSTICS
# ============================================================

if provisional_winner is not None:
    yearly_winner_df = (
        combined_detail_df
        .groupby("year", as_index=False)
        .agg(
            trades=("r", "size"),
            avg_r=("r", "mean"),
            total_r=("r", "sum"),
            win_rate=("win", "mean"),
        )
    )

    market_winner_df = (
        combined_detail_df
        .groupby(["year", "pair"], as_index=False)
        .agg(
            trades=("r", "size"),
            avg_r=("r", "mean"),
            total_r=("r", "sum"),
        )
    )

    outcome_winner_df = (
        combined_detail_df
        .groupby(["year", "outcome"], as_index=False)
        .size()
        .rename(columns={"size": "trades"})
    )
else:
    yearly_winner_df = pd.DataFrame()
    market_winner_df = pd.DataFrame()
    outcome_winner_df = pd.DataFrame()

print("Evolutionary search complete.")


In [ ]:
# ============================================================
# CELL 2 — RESULTS ONLY
# ============================================================
# Run after Cell 1 and paste this output back into ChatGPT.
# ============================================================

pd.set_option("display.width", 420)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

def show(title, df, rows=30):
    print("\n" + "=" * 130)
    print(title)
    print("=" * 130)

    if df is None or df.empty:
        print("EMPTY")
        return

    print(df.head(rows).to_string(index=False))


show(
    "1. GA CONVERGENCE — LAST 15 GENERATIONS",
    ga_history_df.tail(15),
    rows=20,
)

DISCOVERY_DISPLAY_COLS = [
    "rank",
    "score",
    "tf",
    "decision_minutes",
    "use_close",
    "close_threshold",
    "use_against_ratio",
    "against_ratio",
    "use_consecutive",
    "min_consecutive",
    "require_lost_1p5",
    "require_no_reclaim",
    "stop_type",
    "stop_lookback",
    "atr_buffer",
    "atr_stop_mult",
    "target_r",
    "max_hold_minutes",
    "trades",
    "avg_r",
    "median_year_avg",
    "worst_year_avg",
    "positive_years",
    "years",
    "positive_markets",
    "markets",
    "worst_market_avg",
    "max_dd_r",
    "complexity",
]

show(
    "2. TOP 25 — DISCOVERY SEARCH (2022-2024 ONLY)",
    discovery_top_df[
        [c for c in DISCOVERY_DISPLAY_COLS if c in discovery_top_df.columns]
    ],
    rows=25,
)

VALIDATION_DISPLAY_COLS = DISCOVERY_DISPLAY_COLS + [
    "val_2025_trades",
    "val_2025_avg_r",
    "val_2025_total_r",
    "val_2025_dd_r",
    "val_2025_positive_markets",
    "validation_score",
]

show(
    "3. TOP 25 — AFTER 2025 CHRONOLOGICAL VALIDATION",
    validated_df[
        [c for c in VALIDATION_DISPLAY_COLS if c in validated_df.columns]
    ],
    rows=25,
)

show(
    "4. ELIGIBLE PROVISIONAL FINALISTS",
    eligible_finalists[
        [c for c in VALIDATION_DISPLAY_COLS if c in eligible_finalists.columns]
    ],
    rows=20,
)

print("\n" + "=" * 130)
print("5. PROVISIONAL WINNER")
print("=" * 130)

if provisional_winner is None:
    print(
        "NO CANDIDATE PASSED THE ROBUSTNESS GATE.\n"
        "Do not rescue the search by loosening filters immediately."
    )
else:
    print(
        provisional_winner[
            [c for c in VALIDATION_DISPLAY_COLS if c in provisional_winner.index]
        ].to_string()
    )

    print("\n2026 HOLDOUT:")
    if holdout_df.empty:
        print("No 2026 trades.")
    else:
        print(
            pd.DataFrame([{
                "trades": len(holdout_df),
                "avg_r": holdout_df["r"].mean(),
                "median_r": holdout_df["r"].median(),
                "win_rate": holdout_df["win"].mean(),
                "total_r": holdout_df["r"].sum(),
                "max_dd_r": max_drawdown_r(holdout_df),
            }]).to_string(index=False)
        )

show(
    "6. WINNER — YEAR BY YEAR",
    yearly_winner_df,
    rows=20,
)

show(
    "7. WINNER — MARKET × YEAR",
    market_winner_df,
    rows=30,
)

show(
    "8. WINNER — OUTCOME COUNTS",
    outcome_winner_df,
    rows=30,
)
